# KFOUND — Video Inference Demo

This notebook demonstrates how to use the trained **class-agnostic Faster R-CNN** detector
(from Stage 4 of the KFOUND pipeline) to perform object detection on video.

## Requirements
- Trained Faster R-CNN model (`model_final.pth`)
- `detectron2`, `opencv-python`, `supervision`
- An input video file

## Usage
1. Set `MODEL_WEIGHTS` to your trained model path
2. Set `VIDEO_PATH` to your input video
3. Run all cells

In [ ]:
import os
import sys

# Ensure repo root is on path (run notebook from examples/ or repo root)
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if os.path.exists(os.path.join(REPO_ROOT, "KFOUND")):
    sys.path.insert(0, REPO_ROOT)
else:
    REPO_ROOT = os.getcwd()  # already at repo root

print(f"Repo root: {REPO_ROOT}")

In [ ]:
# ============================================================
# Configuration — EDIT THESE PATHS
# ============================================================

# Path to the trained Faster R-CNN model
MODEL_WEIGHTS = os.path.join(REPO_ROOT, "weights", "model_final.pth")

# Path to the Detectron2 config
CONFIG_FILE = os.path.join(REPO_ROOT, "configs", "RN50_DINO_FRCNN_COCO20k_CAD.yaml")

# Input video
VIDEO_PATH = "input_video.mp4"

# Output video
OUTPUT_PATH = "output_detected.mp4"

# Detection thresholds
SCORE_THRESH = 0.7
NMS_THRESH = 0.7

In [ ]:
import cv2
import numpy as np
import torch
import supervision as sv
from detectron2.engine import DefaultPredictor
from detectron2.config import get_cfg
from detectron2.layers import get_norm
from detectron2.modeling.roi_heads import ROI_HEADS_REGISTRY, Res5ROIHeads

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

In [ ]:
# Custom ROI head (must be registered before loading model)
# This adds an extra BatchNorm after res5, following LOST/MoCo convention
# for better transfer from self-supervised (DINO) backbones.

@ROI_HEADS_REGISTRY.register()
class Res5ROIHeadsExtraNorm(Res5ROIHeads):
    def _build_res5_block(self, cfg):
        seq, out_channels = super()._build_res5_block(cfg)
        norm_type = cfg.MODEL.RESNETS.NORM
        norm_layer = get_norm(norm_type, out_channels)
        seq.add_module("norm", norm_layer)
        return seq, out_channels

In [ ]:
# Build Detectron2 config and predictor
cfg = get_cfg()
cfg.merge_from_file(CONFIG_FILE)
cfg.MODEL.WEIGHTS = MODEL_WEIGHTS
cfg.MODEL.ROI_HEADS.NAME = "Res5ROIHeadsExtraNorm"
cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = SCORE_THRESH
cfg.MODEL.NMS_THRESH_TEST = NMS_THRESH
cfg.MODEL.ROI_HEADS.NUM_CLASSES = 1  # class-agnostic
cfg.SOLVER.IMS_PER_BATCH = 1

predictor = DefaultPredictor(cfg)
print("Model loaded successfully!")

In [ ]:
# Frame processing callback
def process_frame(frame: np.ndarray, _) -> np.ndarray:
    result = predictor(frame)
    detections = sv.Detections.from_detectron2(result)
    box_annotator = sv.BoxAnnotator(thickness=4)
    frame = box_annotator.annotate(scene=frame, detections=detections)
    return frame

# Process video
assert os.path.exists(VIDEO_PATH), f"Video not found: {VIDEO_PATH}"

sv.process_video(
    source_path=VIDEO_PATH,
    target_path=OUTPUT_PATH,
    callback=process_frame
)

print(f"Done! Output saved to: {OUTPUT_PATH}")